# Algoritmos de optimización - Seminario<br>
Nombre y Apellidos: Juan Carlos Vereda Ruiz y Rigel Chulia Ortega  <br>
Url: https://github.com/.../03MAIR---Algoritmos-de-Optimizacion---2019/tree/master/SEMINARIO<br>
Problema:
> 1. Sesiones de doblaje <br>
>2. Organizar los horarios de partidos de La Liga<br>
>3. Combinar cifras y operaciones

Descripción del problema:(copiar enunciado)

Descripción del problema:(copiar enunciado)
Problema 3. Combinar cifras y operaciones
• El problema consiste en analizar el siguiente problema y diseñar un algoritmo que lo resuelva.
• Disponemos de las 9 cifras del 1 al 9 (excluimos el cero) y de los 4 signos básicos de las
operaciones fundamentales: suma(+), resta(-), multiplicación(*) y división(/)
• Debemos combinarlos alternativamente sin repetir ninguno de ellos para obtener una
cantidad dada. Un ejemplo sería para obtener el 4:
4+2-6/3*1 = 4

 Debe analizarse el problema para encontrar todos los valores enteros posibles planteando las
siguientes cuestiones:- ¿Qué valor máximo y mínimo se pueden obtener según las condiciones del problema?- ¿Es posible encontrar todos los valores enteros posibles entre dicho mínimo y máximo ?
• Nota: Es posible usar la función de python “eval” para evaluar una expresión

(*) La respuesta es obligatoria





                                        

# Aux

In [1]:
# Librerias utilizadas

from itertools import permutations
import random
import time
from fractions import Fraction

In [2]:
# Funcion Auxiliar para el calculo de las espresiones.
#Evaluación exacta con precedencia usando Fraction

#Esta función evalúa d1 op1 d2 op2 d3 op3 d4 op4 d5 sin usar eval

OPS = ['+', '-', '*', '/']

def eval_nueva(digits, ops):
    """
    Evalúa con precedencia estándar usando el truco:
      total acumula sumas/restas de términos,
      term acumula el término actual con * y /.
    Todo con Fraction para exactitud.
    """
    # Inicializamos
    term = Fraction(digits[0], 1)
    total = Fraction(0, 1)

    for i, op in enumerate(ops):
        x = Fraction(digits[i+1], 1)

        if op == '*':
            term *= x
        elif op == '/':
            term /= x  # seguro: x != 0
        elif op == '+':
            total += term
            term = x
        elif op == '-':
            total += term
            term = -x
        else:
            raise ValueError(f"Operador no soportado: {op}")

    return total + term

# Ejemplo del enunciado: 2*3+1/4-8 = -5.75
print(eval_nueva([2,3,1,4,8], ['*','+','/','-']))

-7/4


# (*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>








Primero definimos las restricciones del problema:
1. **Restricción de Dígitos**

    •	Solo cifras del 1 al 9: Se excluye explícitamente el número 0.
    •	Sin repetición cifras: solo se puede usar 5 cifras de las posible y no se pueden repetir.
2. **Restricción de Operadores**

    •	Operadores básicos: Solo se pueden usar los cuatro signos fundamentales: suma (+), resta (-), multiplicación (*) y división (/).
    •	Sin repetición de operadores: Cada uno de los 4 operadores debe aparecer exactamente una vez en la expresión.
3. **Restricción de Estructura**

    •	Combinación alternativa: Los elementos deben colocarse intercalados. Esto obliga a que la estructura sea siempre: Número → Operador → Número → Operador → Número → Operador → Número → Operador → Número
4. **Restricción de Resultados**

    • Valores enteros: Aunque el cálculo intermedio (especialmente la división) pueda generar decimales, el enunciado pregunta por los valores enteros posibles.


Calculamos las posibilidades sin esas restriciones:
1. Posiciones de los números: Disponemos de 9 dígitos (1-9) para 5 huecos. Al permitir la repetición, cada hueco tiene 9 opciones posibles:$$9 \times 9 \times 9 \times 9 \times 9 = 9^5 = 59,049 \text{ combinaciones de números}$$ Posiciones de los operadores: Disponemos de 4 operadores ($+, -, \times, \div$) para 4 huecos. Si permitimos que se repitan (por ejemplo, usar siempre la suma), cada hueco tiene 4 opciones:$$4 \times 4 \times 4 \times 4 = 4^4 = 256 \text{ combinaciones de operadores}$$ Total de casos posibles: El total de expresiones distintas que el algoritmo tendría que evaluar se obtiene multiplicando ambos resultados:$$\textbf{Total} = 59,049 \times 256 = \mathbf{15,116,544} \text{}$$

# ¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.

Ahora apalicando las restricciones nos quedaria:

Calculamos las permutaciones de 9 elementos tomados de 5 en 5:$$P(9, 5) = \frac{9!}{(9-5)!} = 9 \times 8 \times 7 \times 6 \times 5 = 15,120$$Posiciones de los operadoresTenemos 4 operadores ($+, -, \times, \div$) para 4 huecos. Esto es una permutación simple de 4:$$P_4 = 4! = 4 \times 3 \times 2 \times 1 = 24$$Total de casos posiblesEl total de combinaciones se obtiene mediante el producto de ambas posibilidades:$$15,120 \times 24 = 362,880 \text{ combinaciones}$$

# Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


La estructura de datos que mejor se adapta al problema depende de dos necesidades:
1. Representar de forma ordenada una expresión candidata
2. Almacenar los resultados obtenidos evitando duplicados.

Para representar cada operación, lo más adecuado es usar **tuplas o listas**: una tupla/lista de 5 dígitos (d1,d2,d3,d4,d5) y una tupla/lista de 4 operadores (op1,op2,op3,op4). Esto es apropiado porque el orden importa y además itertools.permutations devuelve precisamente tuplas, lo que simplifica la generación de combinaciones, que utilizaremos para el caso de la fuerza bruta.

Para almacenar los resultados enteros distintos se utiliza un **conjunto (set)**, ya que elimina duplicados automáticamente y permite comprobar pertenencia de forma eficiente. Esto facilita el análisis posterior (obtener mínimo, máximo y comprobar si el intervalo [min,max] está completo).

Para evaluar con exactitud (especialmente por las divisiones) emplearemos el tipo **Fraction**, que evita errores de redondeo y permite comprobar si un resultado es entero verificando que su denominador sea 1.

Opcionalmente, si podria utilizar un **diccionario (dict)** para guardar cada operacion valida con su resultado.

# Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?




Dado un conjunto de cifras $D = \{1, 2, \dots, 9\}$ y un conjunto de operadores $O = \{+, -, *, /\}$, la función objetivo $f$ se define sobre una secuencia combinada $S = (d_1, o_1, d_2, o_2, d_3, o_3, d_4, o_4, d_5)$ tal que:$$f(S) = (((d_1 \ o_1 \ d_2) \ o_2 \ d_3) \ o_3 \ d_4) \ o_4 \ d_5$$

Donde:Cada $d_i$ es una cifra única de $D$ (sin repetir).

Cada $o_i$ es un operador único de $O$ (sin repetir).

La evaluación de la función $f(S)$ no se realiza de manera puramente secuencial, sino que sigue la jerarquía de operaciones aritmética estándar (prioridad de multiplicación y división sobre suma y resta). Técnicamente, el algoritmo descompone la secuencia en "términos" que se consolidan en un "total" acumulado solo cuando se encuentra un operador de menor jerarquía ($+$ o $-$), garantizando así la precisión matemática del resultado.


# (*)¿Es un problema de maximización o minimización?



No se trata de un problema de optimización clásico, ya que no se busca maximizar ni minimizar un valor concreto. El objetivo es explorar todo el espacio de soluciones para encontrar todos los resultados enteros posibles.

Por tanto, puede considerarse un problema de enumeración o exploración del espacio de soluciones más que un problema de maximización o minimización.

# Diseña un algoritmo para resolver el problema por fuerza bruta

In [3]:
# Conjunto de Datos del enunciado
OPS = ['+', '-', '*', '/']
DIGITOS =[1,2,3,4,5,6,7,8,9]

def Fuerza_Bruta(guardar_validas=True):
    op_perms = list(permutations(OPS, 4)) # creamos una lista que tiene todas las permutaciones de los operadores

    S = set() # Set para almacenar las soluciones vacio
    operaciones_validas = {}  # entero -> string expresión

    tiempo_inicial = time.perf_counter()  # Para el calculo de los tiempos de ejecucion
    count = 0

    for digs in permutations(DIGITOS, 5): # Creamos todas las permutacciones posibles con los digitos
        for ops in op_perms:
            val = eval_nueva(digs, ops)
            count += 1

            if val.denominator == 1: # Comprobamos que la operacion es entera si el denominador es 1
                iv = int(val)
                S.add(iv)
                if guardar_validas and iv not in operaciones_validas:
                    expr = f"{digs[0]}{ops[0]}{digs[1]}{ops[1]}{digs[2]}{ops[2]}{digs[3]}{ops[3]}{digs[4]}"
                    operaciones_validas[iv] = expr

    tiempo_final = time.perf_counter()
    return S, operaciones_validas, count, (tiempo_final - tiempo_inicial)


S_brute, ex_brute, n_expr, t_brute = Fuerza_Bruta(guardar_validas=True)

print("EXPRESIONES EVALUADAS:", n_expr)
print("TIEMPO (s):", round(t_brute, 4))
print("ENTEROS DISTINTOS |S|:", len(S_brute))
print("MIN:", min(S_brute), "MAX:", max(S_brute))

EXPRESIONES EVALUADAS: 362880
TIEMPO (s): 3.648
ENTEROS DISTINTOS |S|: 147
MIN: -69 MAX: 77


### Calcula la complejidad del algoritmo por fuerza bruta

En el caso de tu algoritmo de cifras y operadores (donde $n=9$, $m=4$ y $k=5$):

Como $n$ y $m$ son constantes (valores fijos de 9 y 4), la complejidad Big O técnica para esa ejecución específica es $O(1)$, ya que el número de pasos no crece con la entrada (siempre son 362,880).

Sin embargo, para $n$ dígitos y $m$ operadores, la complejidad sería:$$\mathbf{O(n^k \times m!)}$$(Donde $k$ es el número de posiciones a llenar).

# (*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

In [4]:
def backtracking(DIGITOS,guardar_validas=True):
    S = set() # Set para almacenar las soluciones enteras (sin repeiciones)
    validos = {} #Diccionario para guardar una expresión de ejemplo para cada resultado
    count = 0 # Contador de las combinaciones encontradas

    tiempo_inicial = time.perf_counter() # Iniciamos el cronometro

    # Funcion recursiva Busqueda en profundidad (DFS)
    def dfs(used_digits, used_ops, pos, total, term, expr_str):
        nonlocal count

        if pos == 5: # Cundo ya tenemos los 5 digitos puestos terminamos la Recursividad (caso Base)
            val = total + term # Sumamos el ultimo termino
            count += 1
            # Comprobamos si el valor es un numero entero
            if val.denominator == 1:
                iv = int(val)
                S.add(iv)
                # Si no esta en los ejemplos guardamos el valor
                if guardar_validas and iv not in validos:
                    validos[iv] = expr_str
            return

        for op in OPS: # Recorremos los operadores
            if op in used_ops:
                continue # confirmamos que el operador seleccionado no esta ya puesto

            for d in DIGITOS: # Recorremos los digitos
                if d in used_digits:
                    continue # Comprobamos que no esta colocado el digito seleccionado

                x = Fraction(d, 1)
                new_total = total
                new_term = term

                if op == '*':
                    new_term = term * x
                elif op == '/':
                    new_term = term / x
                elif op == '+':
                    new_total = total + term
                    new_term = x
                elif op == '-':
                    new_total = total + term
                    new_term = -x

                used_digits.add(d) # Añadimos a los digitos usados el digito
                used_ops.add(op) # Añadimos a los operadores usados el operador colocado
                # RECURSIVIDAD
                dfs(used_digits, used_ops, pos + 1, new_total, new_term, expr_str + f"{op}{d}")

                # DESHACER CAMBIOS: Limpiar para la siguiente iteración del bucle
                used_digits.remove(d)
                used_ops.remove(op)

    # Bucle Inicial
    for d1 in DIGITOS:
        dfs({d1}, set(), pos=1, total=Fraction(0,1), term=Fraction(d1,1), expr_str=str(d1))

    tiempo_final = time.perf_counter() # Paramos el cronometro

    return S, validos, count, (tiempo_final - tiempo_inicial)
    """Devuelve:
      S= Set con las soluciones enteras
      validos= Diccionario con los ejemplos
      count= Numero de combinaciones encontradas
      t= Tiempo de ejecucion
    """

# Conjunto de Datos del enunciado
OPS = ['+', '-', '*', '/']
DIGITOS =[1,2,3,4,5,6,7,8,9]

# Inicializacion del Backtraking
S_bt, ex_bt, n_expr_bt, t_bt = backtracking(DIGITOS,guardar_validas=True)

print("EXPRESIONES EVALUADAS (Backtracking):", n_expr_bt)
print("TIEMPO (s):", round(t_bt, 4))
print("ENTEROS DISTINTOS |S|:", len(S_bt))
print("MIN:", min(S_bt), "MAX:", max(S_bt))
print("Numero de Casos revisados ")



EXPRESIONES EVALUADAS (Backtracking): 362880
TIEMPO (s): 1.2824
ENTEROS DISTINTOS |S|: 147
MIN: -69 MAX: 77
Numero de Casos revisados 


Ambos códigos parecen hacer lo mismo, probar todas las combinaciones, pero el enfoque de Backtracking tiene ventajas estructurales y de eficiencia sobre la Fuerza Bruta convencional.

La Fuerza Bruta utiliza permutations(OPS, 4) y permutations(DIGITOS, 5).


*   **Problema**: Esto crea objetos en memoria (o iteradores pesados) que contienen todas las permutaciones de antemano. Si el conjunto de datos creciera, podrías sufrir un desbordamiento de memoria.
*   **Mejora del Backtracking**: No almacena las permutaciones. Las construye y destruye dinámicamente sobre la pila de recursión. Solo guarda lo que necesita para el paso actual, lo que es mucho más eficiente en términos de recursos.


**Evaluación Incremental (Ahorro de Cálculo)**

**Fuerza Bruta:** Llama a una función eval_stream(digs, ops) al final de cada ciclo. Esto significa que para cada expresión, el programa tiene que empezar a calcular desde el primer número cada vez.

**Backtracking:** El cálculo es acumulativo. Cuando pasas al siguiente nivel de la recursión (dfs), ya llevas el total y el term calculado hasta ese punto.

Si tienes 1 + 2 * 3 y vas a probar el siguiente operador, el Backtracking ya sabe que el resultado parcial es 7. La fuerza bruta volvería a calcular 1 + 2 * 3 para cada nueva opción que agregues al final.


**Reutilización de Estructuras**

El Backtracking utiliza los conjuntos used_digits y used_ops de forma local y los "limpia" (remove) al volver atrás.
Esto evita la creación constante de nuevas listas o tuplas que ocurre en los bucles anidados de la Fuerza Bruta.
El uso de un set para used_digits hace que la comprobación if d in used_digits sea de tiempo constante O(1), mientras que manejar índices en permutaciones suele ser más costoso a nivel de CPU.

# Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

In [5]:
# Genera una lista que contiene N sublistas, cada una con 5 dígitos aleatorios sin repeticiones

def generar_datos(n=10):
    datos = []
    for _ in range(n):
        numeros = random.sample(range(1,10),5) # definimos que sean 5 valores entre el 1 y el 9
        datos.append(numeros)
    return datos

# Prueba generando 3 Arrays
datos = generar_datos(5)
datos

[[7, 8, 6, 9, 3],
 [2, 9, 1, 8, 4],
 [9, 8, 6, 5, 4],
 [3, 4, 5, 2, 7],
 [1, 5, 3, 6, 9]]

# Aplica el algoritmo al juego de datos generado

Respuesta

In [6]:
resultados_por_caso = [] #Diccionario para guardar los resultados de cada resultado de los datos

# For para recorrer todas las sublista que tiene datos
for nums in datos:
    S_bt, ex_bt, n_expr_bt, t_bt = backtracking(nums)
    resultados_por_caso.append({
        "numeros": nums,
        "n_enteros": len(S_bt),
        "min": min(S_bt),
        "max": max(S_bt),
        "tiempo": round(t_bt, 4)
    })

resultados_por_caso

[{'numeros': [7, 8, 6, 9, 3],
  'n_enteros': 38,
  'min': -63,
  'max': 77,
  'tiempo': 0.0188},
 {'numeros': [2, 9, 1, 8, 4],
  'n_enteros': 37,
  'min': -69,
  'max': 74,
  'tiempo': 0.0175},
 {'numeros': [9, 8, 6, 5, 4],
  'n_enteros': 23,
  'min': -47,
  'max': 57,
  'tiempo': 0.0214},
 {'numeros': [3, 4, 5, 2, 7],
  'n_enteros': 16,
  'min': -30,
  'max': 36,
  'tiempo': 0.0186},
 {'numeros': [1, 5, 3, 6, 9],
  'n_enteros': 32,
  'min': -46,
  'max': 56,
  'tiempo': 0.018}]

# Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Documentación General:
Python Software Foundation. (s.f.). Python 3.12.0 documentation. Recuperado el 6 de marzo de 2026, de https://docs.python.org/3/


Módulo itertools:
Python Software Foundation. (s.f.). itertools — Functions creating iterators for efficient looping. Python 3.12.0 documentation. https://docs.python.org/3/library/itertools.html


Módulo fractions:
Python Software Foundation. (s.f.). fractions — Rational numbers. Python 3.12.0 documentation. https://docs.python.org/3/library/fractions.html


Peleato, J. J. (s.f.). Backtracking. Algoritmia. https://docs.jjpeleato.com/algoritmia/backtracking


Algoritmos de Optimización:
Jose Manuel Camacho Camacho. (2026). Material y apuntes de la asignatura de Algoritmos de Optimización.Universidad Internacional de Valencia(VIU), Plataforma Virtual.


Python para IA:
Franklin Alvarez Cardinale. (2026). Material y apuntes de la asignatura de Python para la Inteligencia Artificial. Universidad Internacional de Valencia(VIU), Plataforma Virtual.

# Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Aunque el algoritmo actual funciona muy bien para 9 dígitos , si añadieramos muchas cifras, dejaria de ser una opcion.  En ese caso, podríamos usar:

Paralelización: Como cada número inicial abre una rama independiente, podríamos repartir el trabajo entre varios núcleos del procesador para ganar velocidad.

Heurísticas o Algoritmos Genéticos: En lugar de probarlo todo, podríamos usar algoritmos que "aprendan" a combinar los números para acercarse a un valor objetivo, como si estuviéramos evolucionando la mejor solución.

Memoización: Si el problema permitiera repetir números, guardaríamos resultados de trozos de la operación ya calculados para no repetir el mismo trabajo dos veces.

Como hemos comentado , si pasamos de 9 dígitos a $N$ dígitos, nos enfrentaríamos a una explosión combinatoria. Para no morir en el intento, se podrían aplicar técnicas de Branch and Bound (Ramificación y Poda) a costa de no recorrer todos los valores enteros posibles entre el maximo y el minimo. El objetivo ahora seria obtener solo los maximos y minimos. A continuacion se presenta un algoritmo de poda. En el momento que la operación da un numero decimal, se descarta y ya no se continua por esa rama. Se puede comprobar que el tiempo que necesita para terminar y los nodos visitados son significativamente muchos menos.

In [7]:
# Conjunto de Datos del enunciado
OPS = ['+', '-', '*', '/']
DIGITOS = [1, 2, 3, 4, 5, 6, 7, 8, 9]


def backtracking_con_poda(DIGITOS, guardar_validos=True):
    S = set()
    validos = {}
    count = 0

    tiempo_inicial = time.perf_counter()

    def dfs(used_digits, used_ops, pos, total, term, expr_str):
        nonlocal count

        if pos == 5:
            val = total + term
            count += 1
            # Aquí val siempre será entero si aplicamos la poda correctamente antes
            iv = int(val)
            S.add(iv)
            if guardar_validos and iv not in validos:
                validos[iv] = expr_str
            return

        for op in OPS:
            if op in used_ops:
                continue

            for d in DIGITOS:
                if d in used_digits:
                    continue

                x = Fraction(d, 1)
                new_total = total
                new_term = term

                if op == '*':
                    new_term = term * x
                elif op == '/':
                    new_term = term / x
                    # --- PODA---
                    # Si el nuevo término no es entero, abortamos esta rama
                    if new_term.denominator != 1:
                        continue
                elif op == '+':
                    new_total = total + term
                    new_term = x
                elif op == '-':
                    new_total = total + term
                    new_term = -x

                # Si llegamos aquí, la rama sigue siendo válida (o es entero o es +,-,*)
                used_digits.add(d)
                used_ops.add(op)

                #RECURSIVIDAD
                dfs(used_digits, used_ops, pos + 1, new_total, new_term, expr_str + f"{op}{d}")

                # BACKTRACKING
                used_digits.remove(d)
                used_ops.remove(op)

    for d1 in DIGITOS:
        dfs({d1}, set(), pos=1, total=Fraction(0,1), term=Fraction(d1,1), expr_str=str(d1))

    tiempo_final = time.perf_counter()

    return S, validos, count, (tiempo_final - tiempo_inicial)

# Ejecución
S_bt, ex_bt, n_expr_bt, t_bt = backtracking_con_poda(DIGITOS, guardar_validos=True)

print("NODOS VISITADOS (Con Poda):", n_expr_bt)
print("TIEMPO (s):", round(t_bt, 4))
print("ENTEROS DISTINTOS |S|:", len(S_bt))
print("MIN:", min(S_bt), "MAX:", max(S_bt))
print("Numero de validas",len(ex_bt))

NODOS VISITADOS (Con Poda): 80280
TIEMPO (s): 0.4645
ENTEROS DISTINTOS |S|: 147
MIN: -69 MAX: 77
Numero de validas 147
